# Runtime Analysis: Fleet Composition

This notebook analyzes the runtime performance of the fleet composition experiments, specifically focusing on the Mixed Fleet (SCV + MCV) scenarios.

## Objectives
1. Understand how runtime scales with the number of customers.
2. Identify which components of the algorithm consume the most time.
3. Explain the discrepancy between `total_runtime_sec` and the `global` span.
4. Breakdown runtime by algorithmic phases (Clustering, Optimization, Post-optimization).

In [ ]:
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Setup plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Data

We load all JSON files generated by the `run_grid_mixed.py` experiment.

In [ ]:
results = []
files = glob.glob("*.json")
print(f"Found {len(files)} JSON files.")

for f in files:
    try:
        with open(f) as json_file:
            data = json.load(json_file)
            results.append(data)
    except Exception as e:
        print(f"Error loading {f}: {e}")

df = pd.DataFrame(results)
print(f"Loaded {len(df)} valid runs.")

# Preview
if not df.empty:
    print(df[['instance', 'num_customers', 'fleet_type', 'total_runtime_sec']].head())
else:
    print("No data loaded.")

## 2. Extract Time Spans

The `time_measurements` field contains detailed spans. We flatten this into columns for easier analysis.

In [ ]:
def extract_spans(row):
    measurements = row.get('time_measurements', [])
    if not measurements:
        return pd.Series(dtype=float)
    
    spans = {}
    for m in measurements:
        # Use wall_time for analysis
        spans[m['span_name']] = m['wall_time']
    return pd.Series(spans)

if not df.empty:
    span_df = df.apply(extract_spans, axis=1)
    # Combine with original metrics
    df_full = pd.concat([df, span_df], axis=1)

    # Fill NaNs with 0 for spans that didn't run (e.g., phase 2 skipped)
    span_cols = span_df.columns
    df_full[span_cols] = df_full[span_cols].fillna(0.0)

    print("Available spans:", list(span_cols))
else:
    df_full = pd.DataFrame()
    print("DataFrame is empty, cannot extract spans.")

## 3. Total Runtime vs Number of Customers

We use the `global` span as the true total runtime, as it wraps the entire execution.

In [ ]:
if 'global' in df_full.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df_full, 
        x='num_customers', 
        y='global', 
        hue='fleet_type', 
        alpha=0.7
    )
    plt.title('Total Runtime vs Number of Customers')
    plt.xlabel('Number of Customers')
    plt.ylabel('Runtime (seconds)')
    plt.legend(title='Fleet Type')
    plt.show()
else:
    print("Global span not found in data.")

### Analysis of Growth
- The plot above shows how runtime scales with problem size.
- Look for exponential vs linear trends.
- Mixed fleet runs might take longer due to the two-phase approach (split stops enabled).

## 4. Understanding `total_runtime_sec` vs `global`

The user observed that `total_runtime_sec` is often much larger than `global`. 
This is because `total_runtime_sec` was calculated as the **sum of all measured spans**.

Since spans are nested (e.g., `load_demand` happens *inside* `global`), summing them results in double-counting.
The `global` span is the correct measure of wall-clock time.

In [ ]:
if 'global' in df_full.columns:
    # Calculate the sum explicitly to verify
    df_full['sum_of_spans'] = df_full[span_cols].sum(axis=1)
    
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=df_full['global'], y=df_full['sum_of_spans'])
    
    # Reference lines
    max_val = df_full['global'].max()
    plt.plot([0, max_val], [0, max_val], 'r--', label='1:1 (Identity)')
    plt.plot([0, max_val], [0, 2*max_val], 'g--', label='2:1 (Double Counting)')
    
    plt.title('Sum of All Spans vs Global Span')
    plt.xlabel('Global Span (True Runtime) [s]')
    plt.ylabel('Sum of Spans (Accumulated) [s]')
    plt.legend()
    plt.show()
    
    ratio = (df_full['sum_of_spans'] / df_full['global']).mean()
    print(f"Average Ratio (Sum / Global): {ratio:.2f}")

## 5. Runtime Breakdown by Phase

We aggregate the spans into logical phases:
1. **Clustering**: Generating feasible clusters (`clustering`, `clustering_phase1`, `clustering_phase2`)
2. **Optimization (FSM)**: Solving the set cover problem (`fsm_initial`, `fsm_phase1`, `fsm_phase2`)
3. **Post-Optimization**: Improving solutions (`fsm_post_optimization`, `fsm_post_optimization_phaseX`)
4. **IO/Setup**: Loading data and config (`load_demand`, `vehicle_configuration`)

In [ ]:
# Define mappings
phase_mapping = {
    'Clustering': ['clustering', 'clustering_phase1', 'clustering_phase2'],
    'Optimization (FSM)': ['fsm_initial', 'fsm_phase1', 'fsm_phase2'],
    'Post-Optimization': ['fsm_post_optimization', 'fsm_post_optimization_phase1', 'fsm_post_optimization_phase2'],
    'IO/Setup': ['load_demand', 'vehicle_configuration']
}

if not df_full.empty:
    # Aggregate
    phase_data = pd.DataFrame()
    for phase, cols in phase_mapping.items():
        valid_cols = [c for c in cols if c in df_full.columns]
        if valid_cols:
            phase_data[phase] = df_full[valid_cols].sum(axis=1)
        else:
            phase_data[phase] = 0.0

    # Calculate averages
    avg_times = phase_data.mean().sort_values(ascending=False)

    # Bar Plot
    plt.figure(figsize=(10, 6))
    sns.barplot(x=avg_times.index, y=avg_times.values)
    plt.title('Average Runtime Contribution by Phase')
    plt.ylabel('Time (seconds)')
    plt.xticks(rotation=45)
    plt.show()

    # Pie Chart
    plt.figure(figsize=(8, 8))
    plt.pie(avg_times, labels=avg_times.index, autopct='%1.1f%%', startangle=140)
    plt.title('Runtime Distribution')
    plt.show()

### Interpretation
- **Optimization (FSM)** is typically the bottleneck, as it solves the Set Cover Problem (MILP).
- **Clustering** involves generating feasible routes, which can be expensive but usually scales better than the exact solver.
- **Post-optimization** depends on the complexity of the heuristics applied.

This breakdown helps identify where future optimizations should focus.